In [ ]:
import sys
import os
import subprocess
from pathlib import Path

# Ensure project import path
PROJECT_ROOT = Path('/global/home/hpc5656/SLAM')
sys.path.append(str(PROJECT_ROOT))
# Set working directory so relative paths (e.g., src/config/*.yaml) resolve
os.chdir(str(PROJECT_ROOT))
print('CWD:', Path.cwd())

%matplotlib inline

# Enable autoreload for development (automatically reloads modules when they change)
%load_ext autoreload
%autoreload 2

# CuPy is required - ensure CUDA_PATH and LD_LIBRARY_PATH are set
# This allows the notebook to work even if Jupyter wasn't started with modules loaded
if "CUDA_PATH" not in os.environ:
    print("CUDA_PATH not set, attempting to load modules...")
    try:
        result = subprocess.run(
            'module load cuda/12.2 && env',
            shell=True,
            executable='/bin/bash',
            capture_output=True,
            text=True,
            timeout=10
        )
        if result.returncode == 0:
            for line in result.stdout.split('\n'):
                if '=' in line:
                    key, value = line.split('=', 1)
                    os.environ[key] = value
            print(f"✓ Modules loaded. CUDA_PATH: {os.environ.get('CUDA_PATH', 'Not set')}")
        else:
            print(f"⚠ Could not load modules. Error: {result.stderr}")
            raise RuntimeError(
                "CUDA_PATH not set and could not load modules. "
                "Please run: module load cuda/12.2 before starting Jupyter."
            )
    except Exception as e:
        print(f"✗ Could not load modules: {e}")
        raise RuntimeError(
            f"Failed to load CUDA modules: {e}\n"
            "Please ensure CUDA is loaded before starting Jupyter:\n"
            "  module load cuda/12.2"
        ) from e

# Ensure LD_LIBRARY_PATH includes CUDA library directory for NVRTC (libnvrtc.so.12)
# This is required for CuPy to compile kernels at runtime
# Note: Setting this BEFORE importing CuPy is critical
cuda_path = os.environ.get('CUDA_PATH')
libnvrtc_path = None

if cuda_path:
    # Check both standard location and Compute Canada's targets/x86_64-linux/lib location
    cuda_lib_paths = [
        os.path.join(cuda_path, 'lib64'),  # Standard location
        os.path.join(cuda_path, 'targets', 'x86_64-linux', 'lib'),  # Compute Canada location
    ]
    
    current_ld_path = os.environ.get('LD_LIBRARY_PATH', '')
    ld_paths = current_ld_path.split(':') if current_ld_path else []
    paths_added = []
    
    # Find which paths exist and add them to LD_LIBRARY_PATH
    for cuda_lib_path in cuda_lib_paths:
        if os.path.exists(cuda_lib_path):
            if cuda_lib_path not in ld_paths:
                paths_added.append(cuda_lib_path)
                ld_paths.insert(0, cuda_lib_path)  # Prepend for priority
    
    if paths_added:
        os.environ['LD_LIBRARY_PATH'] = ':'.join(ld_paths)
        print(f"✓ Updated LD_LIBRARY_PATH to include: {', '.join(paths_added)}")
    else:
        # Check if paths were already included
        found_paths = [p for p in cuda_lib_paths if p in ld_paths]
        if found_paths:
            print(f"✓ LD_LIBRARY_PATH already includes CUDA libraries: {', '.join(found_paths)}")
    
    # Find libnvrtc.so.12 and preload it using ctypes
    # This ensures CuPy can find it even if LD_LIBRARY_PATH isn't fully respected
    for cuda_lib_path in cuda_lib_paths:
        if os.path.exists(cuda_lib_path):
            potential_libnvrtc = os.path.join(cuda_lib_path, 'libnvrtc.so.12')
            if os.path.exists(potential_libnvrtc):
                libnvrtc_path = potential_libnvrtc
                print(f"✓ Found libnvrtc.so.12 at: {libnvrtc_path}")
                
                # Preload the library using ctypes so CuPy can find it
                # Use RTLD_GLOBAL to make symbols available to other libraries
                try:
                    import ctypes
                    # Try multiple loading strategies
                    try:
                        # Strategy 1: Load with full path and RTLD_GLOBAL
                        lib = ctypes.CDLL(libnvrtc_path, mode=ctypes.RTLD_GLOBAL)
                        print(f"✓ Preloaded libnvrtc.so.12 using ctypes (RTLD_GLOBAL)")
                    except Exception as e1:
                        # Strategy 2: Try without RTLD_GLOBAL
                        try:
                            lib = ctypes.CDLL(libnvrtc_path)
                            print(f"✓ Preloaded libnvrtc.so.12 using ctypes (standard)")
                        except Exception as e2:
                            raise e1 from e2
                except Exception as e:
                    print(f"⚠ Warning: Could not preload libnvrtc.so.12: {e}")
                    print(f"  CuPy may still work if LD_LIBRARY_PATH is set correctly")
                    print(f"  You may need to restart the Jupyter kernel with:")
                    print(f"    export LD_LIBRARY_PATH={os.path.dirname(libnvrtc_path)}:$LD_LIBRARY_PATH")
                
                # Verify that ctypes can find the library by name (as CuPy will try)
                try:
                    import ctypes.util
                    found_lib = ctypes.util.find_library('nvrtc')
                    if found_lib:
                        print(f"✓ ctypes.util.find_library('nvrtc') found: {found_lib}")
                    else:
                        print(f"⚠ ctypes.util.find_library('nvrtc') returned None")
                        print(f"  This may cause issues. Try loading by name:")
                        try:
                            test_lib = ctypes.CDLL('libnvrtc.so.12')
                            print(f"✓ Successfully loaded libnvrtc.so.12 by name")
                        except Exception as name_err:
                            print(f"✗ Failed to load libnvrtc.so.12 by name: {name_err}")
                            print(f"  You MUST restart the Jupyter kernel with LD_LIBRARY_PATH set")
                except Exception as diag_err:
                    print(f"⚠ Could not run diagnostics: {diag_err}")
                break
    
    if not libnvrtc_path:
        # Try to find any version of libnvrtc.so
        import glob
        for cuda_lib_path in cuda_lib_paths:
            if os.path.exists(cuda_lib_path):
                nvrtc_files = glob.glob(os.path.join(cuda_lib_path, 'libnvrtc.so*'))
                if nvrtc_files:
                    # Try to use the most specific version
                    nvrtc_files.sort(reverse=True)  # Prefer .so.12.2.140 over .so.12 over .so
                    potential_lib = nvrtc_files[0]
                    print(f"⚠ libnvrtc.so.12 not found, but found: {nvrtc_files}")
                    print(f"  Attempting to use: {potential_lib}")
                    try:
                        import ctypes
                        ctypes.CDLL(potential_lib, mode=ctypes.RTLD_GLOBAL)
                        print(f"✓ Preloaded {potential_lib} using ctypes")
                        libnvrtc_path = potential_lib
                    except Exception as e:
                        print(f"⚠ Could not preload {potential_lib}: {e}")
                    break
        else:
            print(f"⚠ Warning: libnvrtc.so.12 not found in any CUDA library directory")
            print(f"  This may cause CuPy kernel compilation to fail")
else:
    print("⚠ CUDA_PATH not set, cannot configure LD_LIBRARY_PATH")

"""
Verification tests for H function (observation probability) in BeliefMDP_n.

Tests:
1. Batched vs sequential H computation performance
2. Vectorized H methods comparison
3. H normalization verification
4. H vs H_log performance comparison
5. Q density unit integration

Uses the same model as T_mat_visuals.ipynb:
- DoubleIntegratorModel with n=4, dt=1.0, max_a=2.0
- LIDAR(fov=360, r_max=10.0, B=8)
"""
# Use CuPy backend for GPU acceleration (falls back to NumPy if not available)
from src.utils.array_backend import np, random, is_cupy
from src.classes.belief_mdp_n import BeliefMDP_n
from src.classes.model import DoubleIntegratorModel, LIDAR
from src.classes.mapping import LidarGridMapVec
from src.utils.map import load_obstacles_config
from tqdm import tqdm
import time

print(f"✓ All imports successful")
print(f"Using backend: {'CuPy (GPU)' if is_cupy else 'NumPy (CPU)'}")

# Verify we're using CuPy
if not is_cupy:
    raise RuntimeError(
        "CuPy is required but not being used. "
        "Check CUDA installation and CuPy setup."
    )

"""
Verification tests for H function (observation probability) in BeliefMDP_n.

Tests:
1. Batched vs sequential H computation performance
2. Vectorized H methods comparison
3. H normalization verification
4. H vs H_log performance comparison
5. Q density unit integration

Uses the same model as T_mat_visuals.ipynb:
- DoubleIntegratorModel with n=4, dt=1.0, max_a=2.0
- LIDAR(fov=360, r_max=10.0, B=8)
"""


CWD: /global/home/hpc5656/SLAM
CUDA_PATH not set, attempting to load modules...
✓ Modules loaded. CUDA_PATH: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2
✓ Updated LD_LIBRARY_PATH to include: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2/lib64, /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2/targets/x86_64-linux/lib
✓ Found libnvrtc.so.12 at: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2/lib64/libnvrtc.so.12
✓ Preloaded libnvrtc.so.12 using ctypes (RTLD_GLOBAL)
✓ ctypes.util.find_library('nvrtc') found: libnvrtc.so.12
✓ Using CuPy for GPU acceleration
✓ Using cupyx.scipy.spatial.KDTree
✓ All imports successful
Using backend: CuPy (GPU)


In [2]:
def test_batched_vs_sequential_H(quantization_level=2, n_samples=1000, batch_sizes=[1, 10, 50, 100, 200]):
    """
    Compare batched vs sequential H computation performance.
    
    Tests:
    1. Sequential H_vectorized_log (one observation at a time)
    2. Batched H_vectorized_log_batch (processes multiple observations together)
    
    Also identifies CPU-GPU transfer bottlenecks.
    
    Args:
        quantization_level: Map quantization level (2 or 3)
        n_samples: Total number of observations to process
        batch_sizes: List of batch sizes to test
    """
    
    obstacles, area = load_obstacles_config(environment='toy2')
    motion_model = DoubleIntegratorModel(p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=0.5, max_a=2.0)
    sensor = LIDAR(fov=360, r_max=10.0, B=8)
    grid_map = LidarGridMapVec(
        x_min=area[0], x_max=area[1],
        y_min=area[2], y_max=area[3],
        quantization_level=quantization_level,
    )
    bmdp = BeliefMDP_n(
        n=quantization_level,
        motion_model=motion_model,
        measurement_model=sensor,
        obstacles=obstacles,
        _map=grid_map,
        sigma_v=1
    )
    bmdp.map.seed_from_obstacles(obstacles)
    
    print(f"\n{'='*70}")
    print(f"=== Batched vs Sequential H Performance Test ===")
    print(f"Quantization level: {quantization_level}")
    print(f"Map size: {quantization_level}x{quantization_level} = {quantization_level**2} cells")
    print(f"Total maps: 2^{quantization_level**2} = {2**(quantization_level**2)}")
    print(f"Total observations: {n_samples}")
    print(f"Batch sizes to test: {batch_sizes}")
    print(f"{'='*70}\n")
    
    # Create uniform prior
    m_n = bmdp.SQ.m_n
    M_size = bmdp.len_M
    π = np.ones((m_n, M_size), dtype=np.float64) / (m_n * M_size)
    u = np.array([0.0, 0.0])
    
    # Generate observations
    print("Generating observations...")
    y_samples = [random.uniform(0.1, sensor.r_max, sensor.B) for _ in range(n_samples)]
    Y_batch_all = np.array(y_samples)  # (n_samples, B) - keep on GPU
    
    results = {}
    
    # Test 1: Sequential processing (baseline)
    print(f"\n--- Testing Sequential H_vectorized_log ---")
    H_seq_values = []
    start_time = time.time()
    for y_sample in tqdm(y_samples, desc="Sequential", leave=False):
        h_val = bmdp.H_vectorized_log(y_sample, π, u)
        H_seq_values.append(float(h_val))
    time_seq = time.time() - start_time
    results['sequential'] = {
        'time': time_seq,
        'time_per_call': time_seq / n_samples * 1000,
        'values': np.array(H_seq_values)
    }
    print(f"  Time: {time_seq:.4f}s ({results['sequential']['time_per_call']:.4f} ms/call)")
    
    # Test 2: Batched processing with different batch sizes
    for batch_size in batch_sizes:
        if batch_size > n_samples:
            continue
            
        print(f"\n--- Testing Batched H_vectorized_log_batch (batch_size={batch_size}) ---")
        H_batch_values = []
        n_batches = (n_samples + batch_size - 1) // batch_size
        
        start_time = time.time()
        for i in tqdm(range(n_batches), desc=f"Batched (size={batch_size})", leave=False):
            start_idx = i * batch_size
            end_idx = min(start_idx + batch_size, n_samples)
            Y_batch = Y_batch_all[start_idx:end_idx]  # (batch_size, B) - stays on GPU
            
            # Process batch - returns GPU array
            H_batch = bmdp.H_vectorized_log_batch(Y_batch, π, u)  # (batch_size,)
            
            # Convert to CPU only once per batch (minimize transfers)
            H_batch_cpu = H_batch.get() if is_cupy else H_batch
            H_batch_values.extend(H_batch_cpu.tolist())
        
        time_batch = time.time() - start_time
        results[f'batched_{batch_size}'] = {
            'time': time_batch,
            'time_per_call': time_batch / n_samples * 1000,
            'time_per_batch': time_batch / n_batches * 1000,
            'values': np.array(H_batch_values)
        }
        speedup = time_seq / time_batch
        print(f"  Time: {time_batch:.4f}s ({results[f'batched_{batch_size}']['time_per_call']:.4f} ms/call)")
        print(f"  Speedup: {speedup:.2f}x vs sequential")
        print(f"  Time per batch: {results[f'batched_{batch_size}']['time_per_batch']:.4f} ms")
    
    # Performance summary
    print(f"\n{'='*70}")
    print("Performance Summary:")
    print(f"{'Method':<30s} {'Time (s)':<15s} {'ms/call':<15s} {'Speedup':<15s}")
    print(f"{'-'*70}")
    baseline_time = results['sequential']['time']
    print(f"{'Sequential':<30s} {baseline_time:<15.4f} {results['sequential']['time_per_call']:<15.4f} {'1.00x':<15s}")
    
    for batch_size in batch_sizes:
        if batch_size > n_samples:
            continue
        key = f'batched_{batch_size}'
        speedup = baseline_time / results[key]['time']
        print(f"{f'Batched (size={batch_size})':<30s} {results[key]['time']:<15.4f} {results[key]['time_per_call']:<15.4f} {f'{speedup:.2f}x':<15s}")
    
    # Numerical agreement check
    print(f"\n{'='*70}")
    print("Numerical Agreement:")
    H_seq_arr = results['sequential']['values']
    
    for batch_size in batch_sizes:
        if batch_size > n_samples:
            continue
        key = f'batched_{batch_size}'
        H_batch_arr = results[key]['values']
        
        # Compare values
        threshold = 1e-100
        non_zero_mask = (H_seq_arr > threshold) | (H_batch_arr > threshold)
        if np.any(non_zero_mask):
            denom = np.maximum(H_seq_arr[non_zero_mask], H_batch_arr[non_zero_mask])
            denom = np.maximum(denom, threshold)
            rel_errors = np.abs(H_seq_arr[non_zero_mask] - H_batch_arr[non_zero_mask]) / denom
            max_rel_error = float(np.max(rel_errors))
            print(f"  Batched (size={batch_size}) vs Sequential: max rel error = {max_rel_error:.6e}")
            if max_rel_error < 1e-10:
                print(f"    ✓ Perfect agreement")
            elif max_rel_error < 1e-6:
                print(f"    ✓ Excellent agreement")
            else:
                print(f"    ⚠ Disagreement: {max_rel_error*100:.2f}%")
    
    # CPU-GPU transfer analysis
    print(f"\n{'='*70}")
    print("CPU-GPU Transfer Analysis:")
    print("  Sequential method:")
    print(f"    - Transfers per call: 1 (H result)")
    print(f"    - Total transfers: {n_samples}")
    print(f"    - Transfer overhead: ~{n_samples * 8 / 1024:.2f} KB (assuming float64)")
    
    for batch_size in batch_sizes:
        if batch_size > n_samples:
            continue
        n_batches = (n_samples + batch_size - 1) // batch_size
        print(f"  Batched (size={batch_size}):")
        print(f"    - Transfers per batch: 1 (H_batch result)")
        print(f"    - Total transfers: {n_batches}")
        print(f"    - Transfer reduction: {n_samples / n_batches:.1f}x fewer transfers")
        print(f"    - Transfer overhead: ~{n_batches * batch_size * 8 / 1024:.2f} KB")
    
    print(f"{'='*70}\n")
    
    return results

# Run test
print("Testing with quantization_level=2 (2x2 maps, 16 total maps)")
results_batch_n2 = test_batched_vs_sequential_H(quantization_level=2, n_samples=1000, batch_sizes=[1, 10, 50, 100, 200, 500])


Testing with quantization_level=2 (2x2 maps, 16 total maps)
Loaded cached T_mat from /global/home/hpc5656/SLAM/cache/T_mat/T_mat_n2_map2x2_max2.0_5c1c430d.npz

=== Batched vs Sequential H Performance Test ===
Quantization level: 2
Map size: 2x2 = 4 cells
Total maps: 2^4 = 16
Total observations: 1000
Batch sizes to test: [1, 10, 50, 100, 200, 500]

Generating observations...

--- Testing Sequential H_vectorized_log ---


  Time: 34.4600s (34.4600 ms/call)

--- Testing Batched H_vectorized_log_batch (batch_size=1) ---


  Time: 24.7832s (24.7832 ms/call)
  Speedup: 1.39x vs sequential
  Time per batch: 24.7832 ms

--- Testing Batched H_vectorized_log_batch (batch_size=10) ---


  Time: 2.7503s (2.7503 ms/call)
  Speedup: 12.53x vs sequential
  Time per batch: 27.5031 ms

--- Testing Batched H_vectorized_log_batch (batch_size=50) ---


  Time: 0.4141s (0.4141 ms/call)
  Speedup: 83.21x vs sequential
  Time per batch: 20.7071 ms

--- Testing Batched H_vectorized_log_batch (batch_size=100) ---


  Time: 0.2089s (0.2089 ms/call)
  Speedup: 164.96x vs sequential
  Time per batch: 20.8897 ms

--- Testing Batched H_vectorized_log_batch (batch_size=200) ---


  Time: 0.1016s (0.1016 ms/call)
  Speedup: 339.09x vs sequential
  Time per batch: 20.3250 ms

--- Testing Batched H_vectorized_log_batch (batch_size=500) ---


  Time: 0.0434s (0.0434 ms/call)
  Speedup: 793.98x vs sequential
  Time per batch: 21.7009 ms

Performance Summary:
Method                         Time (s)        ms/call         Speedup        
----------------------------------------------------------------------
Sequential                     34.4600         34.4600         1.00x          
Batched (size=1)               24.7832         24.7832         1.39x          
Batched (size=10)              2.7503          2.7503          12.53x         
Batched (size=50)              0.4141          0.4141          83.21x         
Batched (size=100)             0.2089          0.2089          164.96x        
Batched (size=200)             0.1016          0.1016          339.09x        
Batched (size=500)             0.0434          0.0434          793.98x        

Numerical Agreement:

CPU-GPU Transfer Analysis:
  Sequential method:
    - Transfers per call: 1 (H result)
    - Total transfers: 1000
    - Transfer overhead: ~7.81 KB (assumin

In [ ]:
def test_vectorized_H_performance(quantization_level=2, n_samples=100):
    """
    Compare performance of vectorized H methods vs H_log.
    
    Tests:
    1. H_log - existing log-space with iteration
    2. H_vectorized - new vectorized version (standard)
    3. H_vectorized_log - new vectorized log-space version
    
    Args:
        quantization_level: Map quantization level (2 or 3)
        n_samples: Number of observations to test
    """
    import time
    
    obstacles, area = load_obstacles_config(environment='toy2')
    motion_model = DoubleIntegratorModel(p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=0.5, max_a=2.0)
    sensor = LIDAR(fov=360, r_max=10.0, B=8)
    grid_map = LidarGridMapVec(
        x_min=area[0], x_max=area[1],
        y_min=area[2], y_max=area[3],
        quantization_level=quantization_level,
    )
    
    print(f"\n{'='*60}")
    print(f"=== Vectorized H Performance Test ===")
    print(f"Quantization level: {quantization_level}")
    print(f"Map size: {quantization_level}x{quantization_level} = {quantization_level**2} cells")
    print(f"Total maps: 2^{quantization_level**2} = {2**(quantization_level**2)}")
    print(f"{'='*60}\n")
    
    bmdp = BeliefMDP_n(
        n=quantization_level,
        motion_model=motion_model,
        measurement_model=sensor,
        obstacles=obstacles,
        _map=grid_map,
        sigma_v=1
    )
    bmdp.map.seed_from_obstacles(obstacles)
    
    # Check if vectorized methods are available
    if bmdp.all_maps_3d is None:
        print(f"⚠ Vectorized methods not available: map space too large")
        print(f"  H*W = {bmdp.map_H * bmdp.map_W} > 16")
        return None
    
    print(f"✓ All maps cached: shape {bmdp.all_maps_3d.shape}")
    print(f"  Memory: {bmdp.all_maps_3d.nbytes / 1024 / 1024:.2f} MB\n")
    
    # Create concentrated belief
    m_n = bmdp.SQ.m_n
    X_ind = m_n // 2
    M_ind = 0
    M_size = bmdp.len_M
    
    π = np.zeros((m_n, M_size), dtype=np.float64)
    π[X_ind, M_ind] = 1.0
    u = np.array([0.0, 0.0])
    
    # Generate test observations
    B = sensor.B
    r_max = sensor.r_max
    y_samples = [random.uniform(0.1, r_max, B) for _ in range(n_samples)]
    
    print(f"Testing with {n_samples} observations...\n")
    
    results = {}
    
    # Test 1: H_log (existing method)
    print("--- Testing H_log (iterative log-space) ---")
    H_log_values = []
    start_time = time.time()
    for y_sample in tqdm(y_samples, desc="H_log", leave=False):
        h_log_val = bmdp.H_log(y_sample, π, u, show_progress=False)
        h_val = np.exp(h_log_val)
        H_log_values.append(float(h_val.item() if hasattr(h_val, 'item') else float(h_val)))
    time_h_log = time.time() - start_time
    H_log_values = np.asarray(H_log_values)
    
    results['H_log'] = {
        'time': time_h_log,
        'time_per_call': time_h_log / n_samples * 1000,
        'values': H_log_values
    }
    print(f"  Time: {time_h_log:.4f}s ({time_h_log/n_samples*1000:.4f}ms/call)")
    print(f"  Mean H: {np.mean(H_log_values):.6e}\n")
    
    # Test 2: H_vectorized (new vectorized standard)
    print("--- Testing H_vectorized (vectorized standard) ---")
    H_vec_values = []
    start_time = time.time()
    for y_sample in tqdm(y_samples, desc="H_vectorized", leave=False):
        h_val = bmdp.H_vectorized(y_sample, π, u)
        H_vec_values.append(float(h_val.item() if hasattr(h_val, 'item') else float(h_val)))
    time_h_vec = time.time() - start_time
    H_vec_values = np.asarray(H_vec_values)
    
    results['H_vectorized'] = {
        'time': time_h_vec,
        'time_per_call': time_h_vec / n_samples * 1000,
        'values': H_vec_values
    }
    print(f"  Time: {time_h_vec:.4f}s ({time_h_vec/n_samples*1000:.4f}ms/call)")
    print(f"  Mean H: {np.mean(H_vec_values):.6e}\n")
    
    # Test 3: H_vectorized_log (new vectorized log-space)
    print("--- Testing H_vectorized_log (vectorized log-space) ---")
    H_vec_log_values = []
    start_time = time.time()
    for y_sample in tqdm(y_samples, desc="H_vectorized_log", leave=False):
        h_log_val = bmdp.H_vectorized_log(y_sample, π, u)
        h_val = np.exp(h_log_val)
        H_vec_log_values.append(float(h_val.item() if hasattr(h_val, 'item') else float(h_val)))
    time_h_vec_log = time.time() - start_time
    H_vec_log_values = np.asarray(H_vec_log_values)
    
    results['H_vectorized_log'] = {
        'time': time_h_vec_log,
        'time_per_call': time_h_vec_log / n_samples * 1000,
        'values': H_vec_log_values
    }
    print(f"  Time: {time_h_vec_log:.4f}s ({time_h_vec_log/n_samples*1000:.4f}ms/call)")
    print(f"  Mean H: {np.mean(H_vec_log_values):.6e}\n")
    
    # Compare results
    print("--- Performance Comparison ---")
    baseline_time = results['H_log']['time']
    
    for method_name, method_results in results.items():
        if method_name == 'H_log':
            continue
        speedup = baseline_time / method_results['time']
        print(f"{method_name:20s}: {speedup:.2f}x faster than H_log")
    
    # Check numerical agreement
    print("\n--- Numerical Agreement ---")
    H_log_arr = results['H_log']['values']
    H_vec_arr = results['H_vectorized']['values']
    H_vec_log_arr = results['H_vectorized_log']['values']
    
    # Compare H_vectorized vs H_log
    threshold = np.asarray(1e-100, dtype=np.float64)
    non_zero_mask = (H_log_arr > threshold) | (H_vec_arr > threshold)
    if np.any(non_zero_mask):
        # np.maximum only takes 2 arguments, so chain the calls
        denom = np.maximum(H_log_arr[non_zero_mask], H_vec_arr[non_zero_mask])
        denom = np.maximum(denom, threshold)
        rel_errors_vec = np.abs(H_log_arr[non_zero_mask] - H_vec_arr[non_zero_mask]) / denom
        max_rel_error_vec = float(np.max(rel_errors_vec))
        print(f"H_vectorized vs H_log: max rel error = {max_rel_error_vec:.6e}")
        if max_rel_error_vec < 0.01:
            print("  ✓ Agreement within 1%")
        else:
            print(f"  ⚠ Disagreement: {max_rel_error_vec*100:.2f}%")
    
    # Compare H_vectorized_log vs H_log
    non_zero_mask_log = (H_log_arr > threshold) | (H_vec_log_arr > threshold)
    if np.any(non_zero_mask_log):
        # np.maximum only takes 2 arguments, so chain the calls
        denom_log = np.maximum(H_log_arr[non_zero_mask_log], H_vec_log_arr[non_zero_mask_log])
        denom_log = np.maximum(denom_log, threshold)
        rel_errors_vec_log = np.abs(H_log_arr[non_zero_mask_log] - H_vec_log_arr[non_zero_mask_log]) / denom_log
        max_rel_error_vec_log = float(np.max(rel_errors_vec_log))
        print(f"H_vectorized_log vs H_log: max rel error = {max_rel_error_vec_log:.6e}")
        if max_rel_error_vec_log < 0.01:
            print("  ✓ Agreement within 1%")
        else:
            print(f"  ⚠ Disagreement: {max_rel_error_vec_log*100:.2f}%")
    
    print(f"\n{'='*60}")
    print("Summary:")
    print(f"  H_log:            {results['H_log']['time_per_call']:.4f} ms/call")
    print(f"  H_vectorized:     {results['H_vectorized']['time_per_call']:.4f} ms/call ({baseline_time/results['H_vectorized']['time']:.2f}x speedup)")
    print(f"  H_vectorized_log: {results['H_vectorized_log']['time_per_call']:.4f} ms/call ({baseline_time/results['H_vectorized_log']['time']:.2f}x speedup)")
    print(f"{'='*60}\n")
    
    return results

# Test with n=2 and n=3
print("Testing with quantization_level=2 (2x2 maps, 16 total maps)")
results_n2 = test_vectorized_H_performance(quantization_level=2, n_samples=100)

print("\n\n" + "="*60 + "\n")

print("Testing with quantization_level=3 (3x3 maps, 512 total maps)")
results_n3 = test_vectorized_H_performance(quantization_level=3, n_samples=100)


In [ ]:
def test_H_normalization():
    """
    Test 2: Verify that H integrates to 1 over observation space.

    H(y|π,u) should satisfy: ∫ H(y|π,u) dy = 1
    """
    obstacles, area = load_obstacles_config(environment='toy2')
    # Use same model as T_mat_visuals.ipynb
    motion_model = DoubleIntegratorModel(p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=0.5, max_a=2.0)
    sensor = LIDAR(fov=360, r_max=10.0, B=8)
    grid_map = LidarGridMapVec(
        x_min=area[0], x_max=area[1],
        y_min=area[2], y_max=area[3],
        quantization_level=2,  # Match n=4 from T_mat_visuals
    )
    bmdp = BeliefMDP_n(
        n=2,  # Match T_mat_visuals
        motion_model=motion_model,
        measurement_model=sensor,
        obstacles=obstacles,
        _map=grid_map,
        sigma_v=1
    )
    bmdp.map.seed_from_obstacles(obstacles)

    # Create a uniform prior over (state, map)
    # Use middle state index for 4D state space
    m_n = bmdp.SQ.m_n
    X_ind = m_n // 2  # Middle state
    M_ind = 0
    M_size = bmdp.len_M

    π = np.zeros((m_n, M_size), dtype=np.float64)
    π[X_ind, M_ind] = 1.0  # Concentrated belief

    u = np.array([0.0, 0.0])  # 2D action for DoubleIntegratorModel

    print(f"\n=== Testing H normalization ===")
    print(f"Belief concentrated at state {X_ind}, map {M_ind}")
    print(f"m_n={m_n}, M_size={M_size}, π shape={π.shape}")
    print(f"π sum: {π.sum():.6f}, π max: {π.max():.6f}")
    
    # DIAGNOSTICS: Check what happens inside H
    print(f"\n=== Diagnostics ===")
    u_idx = bmdp.AQ.get_quantized_index(u)
    print(f"Action u={u}, quantized to index {u_idx}")
    Tn_mat = bmdp.T_mat[:, :, u_idx]
    print(f"T_mat shape: {bmdp.T_mat.shape}, Tn_mat shape: {Tn_mat.shape}")
    print(f"T_mat sum per column (first 5, should be ~1): {Tn_mat.sum(axis=0)[:5]}")
    
    # Compute predicted belief
    integral = Tn_mat @ π  # (m_n, 2^HW)
    print(f"integral shape: {integral.shape}")
    print(f"integral sum: {integral.sum():.6f}, integral max: {integral.max():.6f}")
    print(f"integral[:, {M_ind}] (for map {M_ind}): sum={integral[:, M_ind].sum():.6f}, max={integral[:, M_ind].max():.6f}")
    
    # Test Q function with a sample observation
    y_test = random.uniform(0.1, sensor.r_max, sensor.B)
    print(f"\nTesting Q with sample observation y_test (first 3 values): {y_test[:3]}")
    
    # Get the true map for map index M_ind
    m_test = bmdp.bits_to_map(M_ind, (bmdp.map_H, bmdp.map_W))
    print(f"Map {M_ind} shape: {m_test.shape}, sum: {m_test.sum()}")
    
    Q_vals = bmdp.Q(y_test, bmdp.SQ.X_n, m_test)
    print(f"Q(y_test | X_n, m_{M_ind}) shape: {Q_vals.shape}")
    print(f"Q values: sum={Q_vals.sum():.6e}, max={Q_vals.max():.6e}, min={Q_vals.min():.6e}")
    print(f"Q values > 0: {np.sum(Q_vals > 0)}/{len(Q_vals)}")
    
    # Check what the product would be
    product = Q_vals * integral[:, M_ind]
    print(f"Q * integral[:, {M_ind}]: sum={product.sum():.6e}, max={product.max():.6e}")

    # Sample H over observation space
    B = sensor.B
    r_max = sensor.r_max

    H_values = []
    for _ in tqdm(range(1000), desc="Sampling H values"):
        y_sample = random.uniform(0.1, r_max, B)
        h_val = bmdp.H(y_sample, π, u)
        # Convert CuPy/NumPy scalar to Python float if needed
        H_values.append(float(h_val.item()) if hasattr(h_val, 'item') else float(h_val))

    H_values_arr = np.array(H_values)
    print(f"\n=== Results ===")
    print(f"Mean H(y|π,u): {np.mean(H_values_arr):.6e}")
    print(f"Max H(y|π,u): {np.max(H_values_arr):.6e}")
    print(f"Min H(y|π,u): {np.min(H_values_arr):.6e}")
    print(f"Non-zero H values: {np.sum(H_values_arr > 0)}/{len(H_values_arr)}")
    print(f"Std H(y|π,u): {np.std(H_values_arr):.6e}")
    
    # Check if values are extremely small (numerical underflow concern)
    if np.max(H_values_arr) < 1e-50:
        print(f"\n⚠ WARNING: H values are extremely small (< 1e-50)")
        print(f"  This suggests observations are very far from ideal observations.")
        print(f"  Consider:")
        print(f"    1. Using observations closer to ideal (e.g., add noise to ideal obs)")
        print(f"    2. Increasing sigma_v to make likelihoods less peaked")
        print(f"    3. Checking if the observation space sampling is appropriate")
    
    # H should be non-negative and finite
    assert np.all(H_values_arr >= 0), "H should return non-negative values"
    assert np.all(H_values_arr < np.inf), "H should return finite values"
    assert np.all(np.isfinite(H_values_arr)), "H should return finite values"

    print("✓ H returns valid probability density values")
    
    # Test with observations closer to ideal to verify H works correctly
    print(f"\n=== Testing H with observations near ideal ===")
    # Get ideal observation for the concentrated belief state
    x_test = bmdp.SQ.X_n[X_ind]
    m_test = bmdp.bits_to_map(M_ind, (bmdp.map_H, bmdp.map_W))
    y_ideal = bmdp.ray_casting(x_test[np.newaxis, :2], m_test)[0]
    
    # Add small noise to ideal observation
    noise_scale = 0.1  # Small noise relative to sigma_v=1
    y_near_ideal = y_ideal + random.normal(0, noise_scale, size=y_ideal.shape)
    H_near_ideal = bmdp.H(y_near_ideal, π, u)
    print(f"Ideal observation y_star (first 3): {y_ideal[:3]}")
    print(f"Noisy observation y (first 3): {y_near_ideal[:3]}")
    print(f"H(y_near_ideal | π, u): {H_near_ideal:.6e}")
    
    if H_near_ideal > 1e-10:
        print(f"✓ H returns reasonable values for observations near ideal")
    else:
        print(f"⚠ H still very small even for observations near ideal - check Q computation")
    
    # Test log-space version for comparison
    print(f"\n=== Testing log-space H (logsumexp) ===")
    H_log_near_ideal = bmdp.H_log(y_near_ideal, π, u, show_progress=False)
    H_from_log = np.exp(H_log_near_ideal)
    print(f"H_log(y_near_ideal | π, u): {H_log_near_ideal:.6e}")
    print(f"exp(H_log): {H_from_log:.6e}")
    print(f"Difference (standard - log-space): {abs(H_near_ideal - H_from_log):.2e}")
    if abs(H_near_ideal - H_from_log) / max(H_near_ideal, 1e-100) < 0.01:
        print(f"✓ Log-space and standard H agree (within 1%)")
    else:
        print(f"⚠ Log-space and standard H differ significantly - may indicate numerical issues")
    
test_H_normalization()


In [ ]:
def test_H_vs_H_log_performance():
    """
    Compare performance between H and H_log methods.
    
    Samples 1000 observations and times both methods to compare:
    1. Execution time
    2. Numerical accuracy (results should match)
    3. Memory efficiency (GPU vs CPU transfers)
    """
    import time
    
    obstacles, area = load_obstacles_config(environment='toy2')
    # Use same model as test_H_normalization
    motion_model = DoubleIntegratorModel(p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=0.5, max_a=2.0)
    sensor = LIDAR(fov=360, r_max=10.0, B=8)
    grid_map = LidarGridMapVec(
        x_min=area[0], x_max=area[1],
        y_min=area[2], y_max=area[3],
        quantization_level=2,
    )
    bmdp = BeliefMDP_n(
        n=2,
        motion_model=motion_model,
        measurement_model=sensor,
        obstacles=obstacles,
        _map=grid_map,
        sigma_v=1
    )
    bmdp.map.seed_from_obstacles(obstacles)

    # Create concentrated belief
    m_n = bmdp.SQ.m_n
    X_ind = m_n // 2
    M_ind = 0
    M_size = bmdp.len_M

    π = np.zeros((m_n, M_size), dtype=np.float64)
    π[X_ind, M_ind] = 1.0

    u = np.array([0.0, 0.0])

    print(f"\n=== Performance Comparison: H vs H_log ===")
    print(f"Belief concentrated at state {X_ind}, map {M_ind}")
    print(f"m_n={m_n}, M_size={M_size}, total maps={bmdp.len_M}")
    
    # Sample observations
    B = sensor.B
    r_max = sensor.r_max
    n_samples = 1000
    
    print(f"\nGenerating {n_samples} random observations...")
    y_samples = []
    for _ in range(n_samples):
        y_sample = random.uniform(0.1, r_max, B)
        y_samples.append(y_sample)
    
    print(f"✓ Generated {len(y_samples)} observations")
    
    # Test standard H method
    print(f"\n--- Testing standard H method ---")
    H_values_standard = []
    start_time = time.time()
    
    for i, y_sample in enumerate(tqdm(y_samples, desc="Computing H (standard)")):
        h_val = bmdp.H(y_sample, π, u, show_progress=False)
        H_values_standard.append(float(h_val.item() if hasattr(h_val, 'item') else float(h_val)))
    
    time_standard = time.time() - start_time
    # Convert to backend array (CuPy if using GPU, NumPy otherwise)
    H_values_standard = np.asarray(H_values_standard)
    
    print(f"Standard H: {time_standard:.4f}s total, {time_standard/n_samples*1000:.4f}ms per call")
    print(f"  Mean H: {np.mean(H_values_standard):.6e}")
    print(f"  Max H: {np.max(H_values_standard):.6e}")
    print(f"  Min H: {np.min(H_values_standard):.6e}")
    
    # Test H_log method
    print(f"\n--- Testing H_log method (log-space) ---")
    H_values_log = []
    start_time = time.time()
    
    for i, y_sample in enumerate(tqdm(y_samples, desc="Computing H_log")):
        h_log_val = bmdp.H_log(y_sample, π, u, show_progress=False)
        h_val_from_log = np.exp(h_log_val)
        H_values_log.append(float(h_val_from_log.item() if hasattr(h_val_from_log, 'item') else float(h_val_from_log)))
    
    time_log = time.time() - start_time
    # Convert to backend array (CuPy if using GPU, NumPy otherwise)
    H_values_log = np.asarray(H_values_log)
    
    print(f"H_log: {time_log:.4f}s total, {time_log/n_samples*1000:.4f}ms per call")
    print(f"  Mean H (from log): {np.mean(H_values_log):.6e}")
    print(f"  Max H (from log): {np.max(H_values_log):.6e}")
    print(f"  Min H (from log): {np.min(H_values_log):.6e}")
    
    # Compare results
    print(f"\n--- Comparison ---")
    speedup = time_standard / time_log if time_log > 0 else float('inf')
    print(f"Speedup: {speedup:.2f}x ({'H_log faster' if speedup > 1 else 'H faster'})")
    
    # Check numerical agreement
    # Ensure both arrays are in the same backend format
    H_values_standard = np.asarray(H_values_standard)
    H_values_log = np.asarray(H_values_log)
    
    # For very small values, use relative error with a threshold
    # Use backend array type for threshold to avoid type mismatches
    threshold = np.asarray(1e-100, dtype=np.float64)
    non_zero_mask = (H_values_standard > threshold) | (H_values_log > threshold)
    
    if np.any(non_zero_mask):
        # Extract values using the mask (handles both CuPy and NumPy)
        h_std_masked = H_values_standard[non_zero_mask]
        h_log_masked = H_values_log[non_zero_mask]
        
        # Compute relative errors
        diff = np.abs(h_std_masked - h_log_masked)
        # Use np.maximum with broadcasting - threshold will be broadcast correctly
        denom = np.maximum(np.maximum(h_std_masked, h_log_masked), threshold)
        rel_errors = diff / denom
        
        max_rel_error = float(np.max(rel_errors))
        mean_rel_error = float(np.mean(rel_errors))
        
        print(f"\nNumerical agreement (non-zero values):")
        print(f"  Mean relative error: {mean_rel_error:.6e}")
        print(f"  Max relative error: {max_rel_error:.6e}")
        n_nonzero = int(np.sum(non_zero_mask))
        print(f"  Non-zero values: {n_nonzero}/{len(H_values_standard)}")
        
        if max_rel_error < 0.01:
            print(f"✓ H and H_log agree within 1%")
        elif max_rel_error < 0.1:
            print(f"⚠ H and H_log differ by up to 10% - may indicate numerical precision issues")
        else:
            print(f"✗ H and H_log differ significantly - check implementation")
    else:
        print(f"\n⚠ All values are extremely small (< 1e-100), cannot compute relative error")
        print(f"  Both methods return very small values, which is expected for uniform sampling")
    
    # Check for any NaN or Inf values
    has_nan_standard = bool(np.any(~np.isfinite(H_values_standard)))
    has_nan_log = bool(np.any(~np.isfinite(H_values_log)))
    
    if has_nan_standard:
        n_nan_std = int(np.sum(~np.isfinite(H_values_standard)))
        print(f"⚠ Standard H produced {n_nan_std} non-finite values")
    if has_nan_log:
        n_nan_log = int(np.sum(~np.isfinite(H_values_log)))
        print(f"⚠ H_log produced {n_nan_log} non-finite values")
    if not has_nan_standard and not has_nan_log:
        print(f"✓ Both methods produce finite values")
    
    print(f"\n=== Performance Summary ===")
    print(f"Standard H: {time_standard:.4f}s ({time_standard/n_samples*1000:.4f}ms/call)")
    print(f"H_log:      {time_log:.4f}s ({time_log/n_samples*1000:.4f}ms/call)")
    print(f"Speedup:    {speedup:.2f}x")
    
    return {
        'time_standard': time_standard,
        'time_log': time_log,
        'speedup': speedup,
        'H_values_standard': H_values_standard,
        'H_values_log': H_values_log
    }

test_H_vs_H_log_performance()


In [ ]:
def test_Q_density_unit_integration():
    """
    Test 1: Verify that Q returns probability DENSITY (not measure).

    For a proper density function q(y|x,m), we should have:
    ∫ q(y|x,m) dy = 1

    Since observations are continuous, we can numerically verify this.
    """
    obstacles, area = load_obstacles_config(environment='toy2')
    # Use same model as T_mat_visuals.ipynb
    motion_model = DoubleIntegratorModel(p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=1.0, max_a=2.0)
    sensor = LIDAR(fov=360, r_max=10.0, B=8)
    grid_map = LidarGridMapVec(
        x_min=area[0], x_max=area[1],
        y_min=area[2], y_max=area[3],
        quantization_level=2,  # Match n=4 from T_mat_visuals
    )
    bmdp = BeliefMDP_n(
        n=2,  # Match T_mat_visuals 
        motion_model=motion_model,
        measurement_model=sensor,
        obstacles=obstacles,
        _map=grid_map,
        sigma_v=1
    )
    bmdp.map.seed_from_obstacles(obstacles)

    # Pick a test state and map (use middle state, but adjust index for 4D state space)
    # For 4D states, we need to pick a reasonable index
    mid_idx = bmdp.SQ.m_n // 2
    x_test = bmdp.SQ.X_n[mid_idx]  # Middle state (4D: [x, y, vx, vy])
    m_test = bmdp.map.occupancy_map.data  # True map

    # Sample observation space Y = [0, r_max]^B
    B = sensor.B
    r_max = sensor.r_max
    n_samples_per_dim = 50

    # Create grid in observation space
    y_values = np.linspace(0.1, r_max, n_samples_per_dim)
    total_integral = 0.0
    dy_volume = (r_max / n_samples_per_dim) ** B  # Volume element

    print(f"\n=== Testing Q density integration ===")
    print(f"State: {x_test}, Map shape: {m_test.shape}")
    print(f"Observation space: [0, {r_max}]^{B}")
    print(f"Sampling grid: {n_samples_per_dim}^{B} points")

    # DIAGNOSTICS: Check what's happening
    print(f"\n=== Diagnostics ===")
    print(f"sigma_v (observation noise): {bmdp.σ_v}")
    print(f"cov_y shape: {bmdp.cov_y.shape}")
    print(f"cov_y determinant: {np.linalg.det(bmdp.cov_y)}")
    print(f"cov_y diagonal (first few): {np.diag(bmdp.cov_y)[:5]}")
    
    # Get ideal observation for the test state
    x_test_pos = x_test[:2]  # Position only
    y_star_test = bmdp.ray_casting(x_test_pos[np.newaxis, :], m_test)[0]
    print(f"\nIdeal observation y_star for test state: {y_star_test}")
    print(f"y_star range: [{np.min(y_star_test):.3f}, {np.max(y_star_test):.3f}]")
    
    # Check a few sample observations
    print(f"\nSample observations vs y_star:")
    for i in range(3):
        y_sample = random.uniform(0.1, r_max, B)
        diff = y_sample - y_star_test
        diff_norm_sq = np.dot(diff, diff)
        print(f"  Sample {i}: y={y_sample[:3]}..., ||y-y_star||²={diff_norm_sq:.3f}")
    
    # Numerically integrate Q over observation space
    Q_values = []
    y_star_ideal = bmdp.ray_casting(x_test_pos[np.newaxis, :], m_test)[0]
    
    for i in tqdm(range(100), desc="Sampling Q values"):  # Sample 100 random points
        y_sample = random.uniform(0.1, r_max, B)
        q_val = bmdp.Q(y_sample, x_test[np.newaxis, :], m_test)[0]
        # Convert CuPy/NumPy scalar to Python float if needed
        q_val_float = float(q_val.item()) if hasattr(q_val, 'item') else float(q_val)
        Q_values.append(q_val_float)
        
        # Diagnostic for first few samples
        if i < 3:
            diff = y_sample - y_star_ideal
            diff_norm_sq = np.dot(diff, diff)
            print(f"  Sample {i}: Q={q_val_float:.2e}, ||y-y_star||²={diff_norm_sq:.3f}")

    Q_values_arr = np.array(Q_values)
    print(f"\n=== Results ===")
    print(f"Mean Q(y|x,m): {np.mean(Q_values_arr):.6e}")
    print(f"Max Q(y|x,m): {np.max(Q_values_arr):.6e}")
    print(f"Min Q(y|x,m): {np.min(Q_values_arr):.6e}")
    print(f"Number of zero values: {np.sum(Q_values_arr == 0)}/{len(Q_values_arr)}")
    print(f"Number of non-zero values: {np.sum(Q_values_arr > 0)}/{len(Q_values_arr)}")

    # For multivariate normal: Q(y|x,m) = N(y; g_bar(x,m), Σ_v)
    # The integral over ℝ^B should equal 1
    # We expect some variation due to numerical sampling
    assert np.all(Q_values_arr > 0), "Q should return positive densities"
    assert np.all(Q_values_arr < np.inf), "Q should return finite densities"

    print("✓ Q returns valid probability density values")
    
test_Q_density_unit_integration()
